# Averaged QUADRANT sampling: 
------------------------------------------

I started a new file because the old one was to messy 
1) Collect all the volumes that belog to one partcipant * quadrant 
2) averaage volumes 
3) compute nilearn 2nd level GLM 

In [1]:

import os
import glob
import subprocess
import tempfile
import numpy as np
import nibabel as nib
from nilearn import image as nlimage
from concurrent.futures import ProcessPoolExecutor
import warnings
import re
import pandas as pd
from nilearn.glm import threshold_stats_img
from nilearn.plotting import plot_design_matrix, plot_glass_brain
import matplotlib.pyplot as plt
from nilearn.glm.second_level import SecondLevelModel
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# ===============================================
# Configuration
# ===============================================

vps      = [i for i in range(44) if i not in [32]]   # list of subject IDs
sessions = ["ses-01", "ses-02"]
APs = ["A", "B", "C"]
home = r"/home/malberti/Unix_Folders/SWEEP2/derivatives"

vp_ABC=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 39, 40, 41, 43]
vp_BCA=[18, 19, 20, 21, 23, 24, 22, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 42]


TEMPLATE = "MNI"   # "HCPex" or "MNI"
TEMPLATE_PATHS = {
    "HCPex": r"/home/malberti/Unix_Folders/SWEEP2/Script/DEWEY_v6/Atlases/MNI_icbm_152_Template_pl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz",
    "MNI":   r"/usr/local/fsl/data/standard/MNI152_T1_2mm_brain.nii.gz",
}

TEMPLATE_PATH = TEMPLATE_PATHS[TEMPLATE]

BVAL_TOLERANCE = 50
SMOOTH_FWHM_MM = 6


QUADRANT_indx = {}  # {(AP, shell): {"FIRST_Oct": [...], ...}}

VECTOR = []
APs = ["A", "B", "C"]

for AP in APs:
    dvs_path = f"/home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_{AP}.dvs"
    bvals = os.path.join(home, "sub-04", "ses-01", "dwi", "signal_drift", f"sub-04_ses-01_dwi_eddy_corrected_{AP}_noPA.bval")
    BVALs = np.loadtxt(bvals)

    vectors = []
    with open(dvs_path) as f:
        for line in f:
            match = re.match(r"Vector\[\d+\]\s*=\s*\(([-\d.]+),([-\d.]+),([-\d.]+)\)", line)
            if match:
                vectors.append([float(match.group(1)), float(match.group(2)), float(match.group(3))])

    vectors = np.array(vectors)
    vectors = np.vstack([[0.0, 0.0, 0.0], vectors])
    AP_marks = np.full((len(vectors), 1), AP)

    print(f"[INFO] AP={AP}: parsed {len(vectors)} | {len(BVALs)} vectors from {dvs_path}")

    VECTOR.append(np.column_stack((vectors, BVALs, AP_marks)))

index = np.tile(np.arange(121), 3).reshape(-1, 1)             # collapse the list of (n_i, 4) arrays into one (total_n, 4) array
VECTORs = np.vstack(VECTOR)              # collapse the list of (n_i, 4) arrays into one (total_n, 4) array

VECTORs = np.column_stack([VECTORs, index])  # now (total_n, 5): x, y, z, bval, index


# Get the index 
tolerance = 50
SHELLs = np.unique(np.round(BVALs / tolerance) * tolerance)
N_volumes = 0
MAX_PER_OCT = 8

for shell in SHELLs:
    bSHELL_Vectors = VECTORs[(VECTORs[:, 3].astype(float) > shell - tolerance) & (VECTORs[:, 3].astype(float) < shell + tolerance)]

    FIRST_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
        ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
    ]

    SECOND_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
        ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
    ]
    THIRD_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) > 0)) |
        ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) < 0))
    ]
    FOURTH_Oct = bSHELL_Vectors[
        ((bSHELL_Vectors[:, 0].astype(float) > 0) & (bSHELL_Vectors[:, 1].astype(float) > 0) & (bSHELL_Vectors[:, 2].astype(float) < 0)) |
        ((bSHELL_Vectors[:, 0].astype(float) < 0) & (bSHELL_Vectors[:, 1].astype(float) < 0) & (bSHELL_Vectors[:, 2].astype(float) > 0))
    ]

    QUADRANT_indx[(shell)] = {
        "FIRST_Oct": FIRST_Oct[:],
        "SECOND_Oct": SECOND_Oct[:],
        "THIRD_Oct": THIRD_Oct[:],
        "FOURTH_Oct": FOURTH_Oct[:],
    }

    N_volumes = N_volumes + len(bSHELL_Vectors)
    print(f"[INFO]  shell {shell}: FIRST={len(FIRST_Oct)} SECOND={len(SECOND_Oct)} THIRD={len(THIRD_Oct)} FOURTH={len(FOURTH_Oct)}")
    print(f"[INFO] shell {shell}: {len(bSHELL_Vectors)} volumes/indices")

print(f"[INFO] Grand total across all shells: {N_volumes}")

[INFO] AP=A: parsed 121 | 121 vectors from /home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_A.dvs
[INFO] AP=B: parsed 121 | 121 vectors from /home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_B.dvs
[INFO] AP=C: parsed 121 | 121 vectors from /home/malberti/wks14/temp/FF_DWI_Drift/DWI_plit_C.dvs
[INFO]  shell 0.0: FIRST=0 SECOND=0 THIRD=0 FOURTH=0
[INFO] shell 0.0: 43 volumes/indices
[INFO]  shell 500.0: FIRST=7 SECOND=9 THIRD=8 FOURTH=8
[INFO] shell 500.0: 32 volumes/indices
[INFO]  shell 1000.0: FIRST=22 SECOND=24 THIRD=23 FOURTH=27
[INFO] shell 1000.0: 96 volumes/indices
[INFO]  shell 1500.0: FIRST=16 SECOND=17 THIRD=15 FOURTH=16
[INFO] shell 1500.0: 64 volumes/indices
[INFO]  shell 2000.0: FIRST=17 SECOND=16 THIRD=14 FOURTH=17
[INFO] shell 2000.0: 64 volumes/indices
[INFO]  shell 2800.0: FIRST=14 SECOND=17 THIRD=17 FOURTH=16
[INFO] shell 2800.0: 64 volumes/indices
[INFO] Grand total across all shells: 363


In [ ]:
from nilearn.image import smooth_img

# Merge and check if the order is the same VECTORs indecs
home = r"/home/malberti/Unix_Folders/SWEEP2/Gradients_Stability_Analysis/Volume_MNI"
#vps = [i for i in range(44)]   # list of subject IDs

header = (nib.load(os.path.join(home, f"sub-00_ses-01_A_desc-MNI.nii.gz"))).header
affine = (nib.load(os.path.join(home, f"sub-00_ses-01_A_desc-MNI.nii.gz"))).affine

def load_volume(home, session, vp, items): #So i can run in parallel
    AP = items[4]
    AP = {"A": "B", "B": "C", "C": "A"}[AP] if vp in vp_BCA else AP
    vol = items[5]
    pid = f"sub-{vp:02d}"
    subjid = f"{pid}_{session}"
    DWIs_loaded = nib.load(os.path.join(home, f"{subjid}_{AP}_desc-MNI.nii.gz"))
    DWIs = DWIs_loaded.get_fdata()
    vol_data = DWIs[..., int(vol)]

    vol_img = nib.Nifti1Image(vol_data, affine=DWIs_loaded.affine, header=DWIs_loaded.header)
    vol_smoothed = smooth_img(vol_img, fwhm=6)

    return vol_smoothed.get_fdata()
for shell, quadrant in QUADRANT_indx.items():
    if shell  < 50:
        continue
    print(f"\n================ SHELL {shell} ================")

    for label, vol_indices in quadrant.items():
        print(f"Computing: Quadrant = {label}\n")

        images_ready2_GLM =[]
        for session in sessions:
            for vp in vps:
                if vp == 32:
                    continue

                volumes_per_participants = Parallel(n_jobs=9)(delayed(load_volume)(home, session, vp, items) for items in vol_indices)
                images_ready2_GLM.append(nib.Nifti1Image(np.mean(volumes_per_participants, axis=-1), affine=affine, header=header))
       
        n_pairs = len(vps) # local var, doesn't clobber N_SUBJ
        subjects = [f"sub-{vvol:02d}" for vvol in range(n_pairs)]
        condition_effect = np.hstack(([1] * n_pairs, [-1] * n_pairs))  # ses-01 block, then ses-02 block
        subject_effect = np.vstack((np.eye(n_pairs), np.eye(n_pairs)))
        paired_design_matrix = pd.DataFrame(
            np.hstack((condition_effect[:, np.newaxis], subject_effect)),
            columns=["PORCO_vs_DIO"] + subjects,  # ses-01 first
        )
        #plot_design_matrix(paired_design_matrix)
        #plt.show()
               
        model = SecondLevelModel(n_jobs=2, verbose=0).fit(images_ready2_GLM, design_matrix=paired_design_matrix)
        maps = model.compute_contrast("PORCO_vs_DIO", output_type="all")

        thresholded_map, threshold = threshold_stats_img(
            maps["z_score"], alpha=0.001, cluster_threshold=10, two_sided=True
        )

        # zmap_path = os.path.join(out_dir, f"paired_ttest_zmap_b{int(shell)}_{label}.nii.gz")
        # thresholded_map.to_filename(zmap_path)
        # print(f"[INFO] shell {int(shell)} {label}: salvato {zmap_path}  threshold={threshold:.2f}")

        # --- glass brain plot ---
        display = plot_glass_brain(
            thresholded_map,
            threshold = 2.5,
            colorbar=True,
            plot_abs=False,
            display_mode="ortho",
            title=f"b{int(shell)} {label} | ses-01 vs ses-02 (z>{threshold:.2f})",
        )

        plt.show()            
